In [1]:
import pandas as pd

In [18]:
df = pd.read_csv('./results/raw_shipment_classification_dataset.csv')

In [19]:
df.duplicated().sum()

np.int64(1508)

In [20]:
df.drop_duplicates(inplace=True)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 916 entries, 0 to 2423
Data columns (total 31 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   delay_days                        916 non-null    int64  
 1   promised_transit_days             916 non-null    float64
 2   days_until_eta                    916 non-null    float64
 3   days_since_ship_so_far            916 non-null    float64
 4   lead_time_days                    916 non-null    float64
 5   coo                               916 non-null    object 
 6   scac                              916 non-null    object 
 7   tariff_amount                     916 non-null    float64
 8   ocean_freight                     916 non-null    float64
 9   delivery_terms                    916 non-null    object 
 10  po_shipment_terms                 916 non-null    object 
 11  tariff_type                       916 non-null    object 
 12  total_bcy   

In [22]:
date_time_columns = ['shipped_date', 'eta']
for col in date_time_columns:
    df[f'{col}'] = pd.to_datetime(df[f'{col}'], errors='raise')

df['shipped_date_weekday'] = df['shipped_date'].dt.weekday
df['shipped_date_month'] = df['shipped_date'].dt.month
df['shipped_date_day'] = df['shipped_date'].dt.day

df.drop(columns=date_time_columns, inplace=True)

In [24]:
# 1. Remove 'USD' and any surrounding whitespace
df['total_bcy'] = df['total_bcy'].str.replace('USD', '').str.strip()

# 2. FIX: Remove all thousands separators (commas)
df['total_bcy'] = df['total_bcy'].str.replace(',', '')

# 3. Convert the clean string to float
df['total_bcy'] = df['total_bcy'].astype(float)

In [25]:
import numpy as np
import pandas as pd

numeric_cols = [
    "promised_transit_days",
    "days_until_eta",
    "days_since_ship_so_far",
    "lead_time_days",
    "tariff_amount",
    "ocean_freight",
    "total_bcy",
    "quantity_in",
    "vendor_avg_promised_transit_days",
    "vendor_p50_promised_transit_days",
    "vendor_p90_promised_transit_days",
    "vendor_avg_realized_delay_days",
    "vendor_p50_realized_delay_days",
    "vendor_p90_realized_delay_days",
    "vendor_on_time_rate",
    "vendor_shipments_with_receipt",
]

df = df.copy()

# 1) Split days_until_eta into two features (keep signal for overdue vs remaining)
df["days_to_eta"]   = np.clip(df["days_until_eta"], a_min=0, a_max=None)          # time remaining
df["days_overdue"]  = np.clip(-df["days_until_eta"], a_min=0, a_max=None)         # overdue days
df.drop(columns=["days_until_eta"], inplace=True)

# 2) Columns that must be non-negative: clip & flag corrections
nonneg_cols = [
    "promised_transit_days",
    "days_since_ship_so_far",
    "lead_time_days",
    "tariff_amount",
    "ocean_freight",
    "total_bcy",
    "quantity_in",
    "vendor_avg_promised_transit_days",
    "vendor_p50_promised_transit_days",
    "vendor_p90_promised_transit_days",
    "vendor_avg_realized_delay_days",
    "vendor_p50_realized_delay_days",
    "vendor_p90_realized_delay_days",
    "vendor_shipments_with_receipt",
]

for col in nonneg_cols:
    flag_col = f"{col}__was_negative"
    df[flag_col] = (df[col] < 0).astype("uint8")
    df[col] = np.clip(df[col], a_min=0, a_max=None)

# 3) Probability/rate column: cap to [0, 1] and flag out-of-range
df["vendor_on_time_rate__was_oob"] = (
    (df["vendor_on_time_rate"] < 0) | (df["vendor_on_time_rate"] > 1)
).astype("uint8")
df["vendor_on_time_rate"] = df["vendor_on_time_rate"].clip(lower=0, upper=1)

# 4) Optional: log1p transform for skewed monetary/quantity/count features (use for linear models)
log1p_cols = ["tariff_amount", "ocean_freight", "total_bcy", "quantity_in", "vendor_shipments_with_receipt"]
for col in log1p_cols:
    df[f"{col}__log1p"] = np.log1p(df[col])  # keep original too (trees like raw scale)

    

In [26]:
# Convert total_bcy to numeric
df['total_bcy'] = pd.to_numeric(df['total_bcy'], errors='coerce')


In [27]:
distance_dict = {
    'INDIA': 11000,
    'CHINA': 6000,
    'INDONESIA': 8200,
    'VIETNAM': 6200,
    'ECUADOR': 2100,
    'THAILAND': 8100
}
df['coo'].value_counts()

coo
INDIA        766
CHINA         50
INDONESIA     47
VIETNAM       43
ECUADOR        7
THAILAND       3
Name: count, dtype: int64

In [28]:
df['distance_nm'] = df['coo'].map(distance_dict)
df.drop(columns=['coo','item_product_category'], inplace=True)

In [29]:
df.columns

Index(['delay_days', 'promised_transit_days', 'days_since_ship_so_far',
       'lead_time_days', 'scac', 'tariff_amount', 'ocean_freight',
       'delivery_terms', 'po_shipment_terms', 'tariff_type', 'total_bcy',
       'quantity_in', 'item_sku', 'item_brand', 'item_manufacturer',
       'item_size', 'vendor_name', 'vendor_avg_promised_transit_days',
       'vendor_p50_promised_transit_days', 'vendor_p90_promised_transit_days',
       'vendor_avg_realized_delay_days', 'vendor_p50_realized_delay_days',
       'vendor_p90_realized_delay_days', 'vendor_on_time_rate',
       'vendor_shipments_with_receipt', 'shipped_date_weekday',
       'shipped_date_month', 'shipped_date_day', 'days_to_eta', 'days_overdue',
       'promised_transit_days__was_negative',
       'days_since_ship_so_far__was_negative', 'lead_time_days__was_negative',
       'tariff_amount__was_negative', 'ocean_freight__was_negative',
       'total_bcy__was_negative', 'quantity_in__was_negative',
       'vendor_avg_promised_

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 916 entries, 0 to 2423
Data columns (total 51 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   delay_days                                      916 non-null    int64  
 1   promised_transit_days                           916 non-null    float64
 2   days_since_ship_so_far                          916 non-null    float64
 3   lead_time_days                                  916 non-null    float64
 4   scac                                            916 non-null    object 
 5   tariff_amount                                   916 non-null    float64
 6   ocean_freight                                   916 non-null    float64
 7   delivery_terms                                  916 non-null    object 
 8   po_shipment_terms                               916 non-null    object 
 9   tariff_type                                    

In [31]:
df.to_csv('./results/cleaned_shipment_classification_dataset.csv', index=False)

In [14]:
# df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoder = LabelEncoder()

# Ensure categorical features are strings (clean)
categorical_cols = [
    'scac', 'delivery_terms',
    'po_shipment_terms', 'tariff_type', 'item_brand',
    'item_manufacturer', 
    'item_size', 'vendor_name'
]

for col in categorical_cols:
    df_encoded[col] = df_encoded[col].astype(str).str.strip().replace('', 'Unknown')

for col in categorical_cols:
    df_encoded[col] = label_encoder.fit_transform(df_encoded[col].astype(str))


In [15]:
from sklearn.preprocessing import StandardScaler

columns_to_scale = [
    "tariff_amount",
    "quantity_in",
    "ocean_freight",
]

scaler = StandardScaler()
df_encoded[columns_to_scale] = scaler.fit_transform(df_encoded[columns_to_scale])